# Per-Sample Spike-In Limit-of-Detection Analysis

Deliberately NOT a disease-effect-size power simulation (see `perturb_genes_fold` docstring below) --
this characterizes detection sensitivity/FDR for a KNOWN, deterministic spike-in amount added on
top of each real sample's own observed count and noise floor, analogous to ERCC-style spike-in QC.
Evaluated under two independent HC/HC split designs: LOBO (held-out batches) and in-sample CV
(10-fold, pathologically miscalibrated HC samples excluded beforehand, see the QC step below).

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.Benchmark import db_hit_compare as dc
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.core.marginal_rqr import marginal_nb_rqr
from MixedEffectsModeling.core.shash import shash_transform_to_z
from MixedEffectsModeling.validation.lobo_engine import load_full_data

DETECTION_LIMIT_DIR = config.DETECTION_LIMIT_DIR
DETECTION_LIMIT_DIR.mkdir(exist_ok=True)

N_PERTURB_GENES = 500
LOG2FCS = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
N_BOOTSTRAP = 5
N_FOLDS = 10  # insample_cv only
Q_LEVELS = [0.05, 0.10, 0.20, 0.25]

ALL_SPLIT_METHODS = ["lobo", "insample_cv"]

In [2]:
data = load_full_data()

_meta_rows = []
for bdir in sorted(config.LOBO_MIXED_DIR.iterdir()):
    meta_path = bdir / "meta.json"
    if not meta_path.exists():
        continue
    meta = json.loads(meta_path.read_text())
    n_hc_test = int(sum(meta["test_is_hc"]))
    _meta_rows.append(dict(batch_id=meta["batch_id"], safe_dir=bdir.name, n_hc_test=n_hc_test))
lobo_batches = pd.DataFrame(_meta_rows).sort_values("n_hc_test", ascending=False).reset_index(drop=True)

BATCHES = ["Ward Z et al._Batch_1", "Moufarrej et al._Batch_2", "Moore et al._Batch_1",
          "Chen et al._Batch_2", "Roskams-Hieter B et al._Batch_2"]
print(lobo_batches[lobo_batches.batch_id.isin(BATCHES)])

                          batch_id                         safe_dir  n_hc_test
0            Ward Z et al._Batch_1            Ward_Z_et_al._Batch_1        116
1         Moufarrej et al._Batch_2         Moufarrej_et_al._Batch_2         93
2             Moore et al._Batch_1             Moore_et_al._Batch_1         71
6              Chen et al._Batch_2              Chen_et_al._Batch_2         31
7  Roskams-Hieter B et al._Batch_2  Roskams-Hieter_B_et_al._Batch_2         27


In [3]:
_cache = {}
name2row = {n: i for i, n in enumerate(data["names"])}
hc_meta_global = pd.read_csv(dc.ZDIR / "hc_meta.csv")
Z_hc_global = np.load(dc.ZDIR / "Z_hc_shash.npy")
gene_names_global = pickle.load(open(dc.ZDIR / "gene_names.pkl", "rb"))
gene_pos_global = {g: j for j, g in enumerate(gene_names_global)}
hc_global_rows = np.array([name2row[s] for s in hc_meta_global["sample"]])

# QC: exclude HC samples with a pathologically inflated frac(|Z|>1.96)
_frac_extreme_hc = (np.abs(Z_hc_global) > 1.96).mean(axis=1)
_q1, _q3 = np.percentile(_frac_extreme_hc, [25, 75])
HC_QC_FENCE = _q3 + 1.5 * (_q3 - _q1)
HC_QC_OUTLIER = _frac_extreme_hc > HC_QC_FENCE  # aligned to hc_meta_global / Z_hc_global rows
print(f"HC QC: excluding {HC_QC_OUTLIER.sum()}/{len(HC_QC_OUTLIER)} samples "
      f"with frac(|Z|>1.96) > {HC_QC_FENCE:.3f} (Tukey fence) from insample_cv")


def load_batch(batch_id):
    if batch_id in _cache:
        return _cache[batch_id]
    safe = lobo_batches.set_index("batch_id").loc[batch_id, "safe_dir"]
    bdir = config.LOBO_MIXED_DIR / safe
    meta = json.loads((bdir / "meta.json").read_text())
    gene_names = pickle.load(open(bdir / "gene_names.pkl", "rb"))
    Z_test = np.load(bdir / "Z_test_shash.npy")
    test_is_hc = np.array(meta["test_is_hc"])

    fits = pd.read_csv(bdir / "model_fits.csv").set_index("gene")
    fits = fits[fits["ok"]]
    shash_p = pd.read_csv(bdir / "shash_params.csv").set_index("gene")
    universe = [g for g in fits.index if g in shash_p.index and g in gene_names]  # model-route only

    is_hc, batch, small = data["is_hc"], data["batch"], data["small_hc_batches"]
    tr_idx = np.where(is_hc & (batch != batch_id) & ~np.isin(batch, list(small)))[0]
    scaler = StandardScaler().fit(data["X_raw"][tr_idx])

    hc_test_names = np.array(meta["test_names"])[test_is_hc]
    hc_test_rows = np.array([name2row[n] for n in hc_test_names])

    out = dict(gene_names=gene_names, gene_pos={g: j for j, g in enumerate(gene_names)},
               Z_hc=Z_test[test_is_hc], universe=universe, fits=fits, shash_p=shash_p,
               scaler=scaler, hc_test_rows=hc_test_rows)
    _cache[batch_id] = out
    return out


def mu_alpha_tau2(b, genes, sample_rows):
    Xs = b["scaler"].transform(data["X_raw"][sample_rows])
    Xa = np.column_stack([np.ones(len(sample_rows)), Xs])
    mu, alpha, tau2 = {}, {}, {}
    fits = b["fits"]
    mu_cols = [c for c in fits.columns if c.startswith("mu_coef_")]
    disp_cols = [c for c in fits.columns if c.startswith("disp_coef_")]
    for g in genes:
        row = fits.loc[g]
        mu_coef = row[mu_cols].values.astype(float)
        disp_coef = row[disp_cols].values.astype(float)
        mu[g] = np.clip(np.exp(Xa @ np.nan_to_num(mu_coef, nan=0.0)), 1e-6, 1e8)
        alpha[g] = (np.exp(-Xa @ np.nan_to_num(disp_coef, nan=0.0)) if not np.all(np.isnan(disp_coef))
                    else np.full(len(sample_rows), float(row["trend_alpha"])))
        tau2[g] = float(row["tau2"])
    return mu, alpha, tau2


def perturb_genes_fold(b, genes, sample_rows, log2fc, seed):
    # spike-in / limit-of-detection design: adds a KNOWN, deterministic amount -- the model's
    # expected mean shift mu*(2^log2fc - 1) -- on top of each sample's REAL observed count, the
    # same way an ERCC spike-in adds a known quantity of material to a real sample and asks
    # whether it's recovered against that sample's own real background noise. We are NOT claiming
    # y_pert is a valid draw from NB(mu*fold, alpha) (its variance stays at the real sample's own
    # level, not the larger variance a true NB(mu*fold, alpha) sample would have) -- so this
    # measures a detection-limit/floor, not a disease-effect-size power estimate. Chosen over
    # posterior-predictive resimulation (the field-standard polyester/splatter design) because it
    # better matches cfRNA's actual noise floor, and because filtering to well-expressed genes to
    # avoid this issue would exclude real low-expression disease-relevant transcripts that
    # legitimately spike in disease.
    mu, alpha, tau2 = mu_alpha_tau2(b, genes, sample_rows)
    shash_p = b["shash_p"]
    z_of = {}
    for g in genes:
        y_real = data["Y"][sample_rows, data["gene_col"][g]]
        y_pert = np.round(np.maximum(y_real + mu[g] * (2 ** log2fc - 1), 0))
        z_raw = marginal_nb_rqr(y_pert, mu[g], alpha[g], tau2[g], seed=seed + hash(g) % 9973)
        srow = shash_p.loc[g]
        z = shash_transform_to_z(z_raw, srow.xi, srow.eta, srow.eps, srow.delta) if srow.ok else z_raw
        z_of[g] = np.clip(z, -50, 50)
    return z_of

HC QC: excluding 51/676 samples with frac(|Z|>1.96) > 0.094 (Tukey fence) from insample_cv


In [4]:
def iter_lobo_folds():
    for batch_id in BATCHES:
        b = load_batch(batch_id)
        n_hc = len(b["hc_test_rows"])
        for boot in range(N_BOOTSTRAP):
            rng = np.random.default_rng(1000 * boot + hash(batch_id) % 9973)
            perm = rng.permutation(n_hc)
            idx_A = perm[: n_hc // 2]
            rows_A = b["hc_test_rows"][idx_A]
            yield dict(group_id=batch_id, b=b, rows_A=rows_A, idx_A=idx_A, boot=boot)


_insample_cache = None


def load_insample_model():
    global _insample_cache
    if _insample_cache is not None:
        return _insample_cache

    genes = pickle.load(open(config.ENGINE_MIXED_DIR / "genes.pkl", "rb"))
    summary = pd.read_csv(config.ENGINE_MIXED_DIR / "training_summary.csv").set_index("gene")
    scaler_ins = pickle.load(open(config.ENGINE_MIXED_DIR / "scaler.pkl", "rb"))

    universe = [g for g in summary.index[summary["ok"].fillna(False)] if g in genes and genes[g].mu_coef is not None]
    recs = [genes[g] for g in universe]
    mu_coef_mat = np.nan_to_num(np.stack([r.mu_coef for r in recs]).astype(float), nan=0.0)
    disp_coef_mat = np.stack([r.disp_coef for r in recs]).astype(float)
    has_disp = ~np.all(np.isnan(disp_coef_mat), axis=1)
    disp_coef_mat = np.nan_to_num(disp_coef_mat, nan=0.0)
    tau2_vec = np.array([r.tau2 for r in recs], dtype=float)
    trend_alpha_vec = np.array([r.trend_alpha for r in recs], dtype=float)

    su = summary.loc[universe]
    ok_arr = su["cv_shash_ok"].fillna(False).infer_objects(copy=False).values.astype(bool)
    xi_arr = np.where(ok_arr, su["cv_shash_xi"].fillna(0.0).values, 0.0)
    eta_arr = np.where(ok_arr, su["cv_shash_eta"].fillna(1.0).values, 1.0)
    eps_arr = np.where(ok_arr, su["cv_shash_eps"].fillna(0.0).values, 0.0)
    delta_arr = np.where(ok_arr, su["cv_shash_delta"].fillna(1.0).values, 1.0)

    mu_cols = [f"mu_coef_{i}" for i in range(mu_coef_mat.shape[1])]
    disp_cols = [f"disp_coef_{i}" for i in range(disp_coef_mat.shape[1])]
    fits = pd.DataFrame(mu_coef_mat, columns=mu_cols, index=universe)
    for i, c in enumerate(disp_cols):
        fits[c] = np.where(has_disp, disp_coef_mat[:, i], np.nan)
    fits["trend_alpha"] = trend_alpha_vec
    fits["tau2"] = tau2_vec
    shash_p = pd.DataFrame(dict(xi=xi_arr, eta=eta_arr, eps=eps_arr, delta=delta_arr, ok=ok_arr), index=universe)

    out = dict(gene_names=gene_names_global, gene_pos=gene_pos_global, Z_hc=Z_hc_global,
               universe=universe, fits=fits, shash_p=shash_p, scaler=scaler_ins,
               hc_test_rows=hc_global_rows)
    _insample_cache = out
    return out


def iter_insample_cv_folds():
    b = load_insample_model()
    keep = np.where(~HC_QC_OUTLIER)[0]  # drop pathologically miscalibrated HC before CV-splitting
    n = len(keep)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
    for fold, (_, idx_A_k) in enumerate(kf.split(np.arange(n))):
        idx_A = keep[idx_A_k]
        rows_A = b["hc_test_rows"][idx_A]
        yield dict(group_id=f"insample_fold{fold}", b=b, rows_A=rows_A, idx_A=idx_A, boot=fold)


SPLIT_PROVIDERS = {"lobo": iter_lobo_folds, "insample_cv": iter_insample_cv_folds}

In [5]:
def run_per_sample(split_method):
    cache_path = DETECTION_LIMIT_DIR / f"sweep_persample_fold__{split_method}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    for fold in SPLIT_PROVIDERS[split_method]():
        group_id, b = fold["group_id"], fold["b"]
        rows_A, idx_A, boot = fold["rows_A"], fold["idx_A"], fold["boot"]
        universe = b["universe"]

        for log2fc in LOG2FCS:
            rng = np.random.default_rng(int(1e6 * log2fc) + boot + hash(group_id) % 9973)
            pert_genes = list(rng.choice(universe, min(N_PERTURB_GENES, len(universe)), replace=False))
            z_of = perturb_genes_fold(b, pert_genes, rows_A, log2fc, seed=int(1e6 * log2fc) + boot)

            Z_A = b["Z_hc"][idx_A].copy()
            labels = np.zeros(len(b["gene_names"]), dtype=bool)
            for g in pert_genes:
                j = b["gene_pos"][g]
                Z_A[:, j] = z_of[g]
                labels[j] = True

            finite = np.isfinite(Z_A)
            p = np.where(finite, 2 * norm.sf(np.abs(np.where(finite, Z_A, 0.0))), np.nan)

            for q in Q_LEVELS:
                tp = fp = fn_ = tn = 0
                for i in range(Z_A.shape[0]):
                    fin_i = finite[i]
                    reject_i = np.zeros(Z_A.shape[1], dtype=bool)
                    reject_i[fin_i] = bh_fdr_reject(p[i, fin_i], q=q)
                    tp += int((reject_i & labels).sum())
                    fp += int((reject_i & ~labels).sum())
                    fn_ += int((~reject_i & labels).sum())
                    tn += int((~reject_i & ~labels).sum())
                rows.append(dict(batch=group_id, boot=boot, log2fc=log2fc, q=q,
                                 n_samples=int(Z_A.shape[0]), n_universe=int(Z_A.shape[1]),
                                 n_perturbed=len(pert_genes), tp=tp, fp=fp, fn=fn_, tn=tn))
        print(split_method, group_id, "per-sample done", flush=True)

    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df


def run_per_sample_all_splits():
    parts = []
    for split_method in ALL_SPLIT_METHODS:
        df = run_per_sample(split_method)
        df["split"] = split_method
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


sweep_persample = run_per_sample_all_splits()
sweep_persample

,batch,boot,log2fc,q,n_samples,n_universe,n_perturbed,tp,fp,fn,tn,split
0,Ward Z et al._Batch_1,0,0.0,0.05,58,19824,500,4,280,28996,1120512,lobo
1,Ward Z et al._Batch_1,0,0.0,0.10,58,19824,500,9,509,28991,1120283,lobo
2,Ward Z et al._Batch_1,0,0.0,0.20,58,19824,500,84,10434,28916,1110358,lobo
3,Ward Z et al._Batch_1,0,0.0,0.25,58,19824,500,118,11349,28882,1109443,lobo
4,Ward Z et al._Batch_1,0,1.0,0.05,58,19824,500,10,270,28990,1120522,lobo
...,...,...,...,...,...,...,...,...,...,...,...,...
835,insample_fold9,9,4.0,0.25,62,19858,500,21052,6793,9948,1193403,insample_cv
836,insample_fold9,9,5.0,0.05,62,19858,500,22601,1215,8399,1198981,insample_cv
837,insample_fold9,9,5.0,0.10,62,19858,500,23578,2535,7422,1197661,insample_cv
838,insample_fold9,9,5.0,0.20,62,19858,500,24884,5828,6116,1194368,insample_cv


In [6]:
counts_ps = sweep_persample.groupby(["split", "q", "log2fc"])[["tp", "fp", "fn", "tn"]].sum()
tp, fp, fn, tn = counts_ps["tp"], counts_ps["fp"], counts_ps["fn"], counts_ps["tn"]
summary_ps = pd.DataFrame(dict(
    TP=tp, FP=fp, FN=fn, TN=tn,
    Precision=tp / (tp + fp), Sensitivity=tp / (tp + fn), Specificity=tn / (tn + fp),
    Accuracy=(tp + tn) / (tp + fp + fn + tn), FDR=fp / (tp + fp),
    F1=2 * tp / (2 * tp + fp + fn),
)).round(3)

for sm in ALL_SPLIT_METHODS:
    print(f"per-sample (no aggregation) -- split method: {sm}")
    display(summary_ps.loc[sm])

null_calib_ps = sweep_persample[sweep_persample.log2fc == 0].groupby(["split", "q"])["fp"].agg(["mean", "max"])
print("null (log2fc=0) per-sample FP calibration:")
display(null_calib_ps)

per-sample (no aggregation) -- split method: lobo


TP      FP      FN        TN  Precision  Sensitivity  \
q    log2fc                                                             
0.05 0.0        293   68892  417207  16066648      0.004        0.001   
     1.0       1581   84264  415919  16051276      0.018        0.004   
     2.0      50922   90229  366578  16045311      0.361        0.122   
     3.0     148966   97365  268534  16038175      0.605        0.357   
     4.0     226696  101682  190804  16033858      0.690        0.543   
     5.0     287805  104889  129695  16030651      0.733        0.689   
0.10 0.0        717   95259  416783  16040281      0.007        0.002   
     1.0       2538   96512  414962  16039028      0.026        0.006   
     2.0      62858  107778  354642  16027762      0.368        0.151   
     3.0     164399  119359  253101  16016181      0.579        0.394   
     4.0     241016  127291  176484  16008249      0.654        0.577   
     5.0     302393  132959  115107  16002581      0.695        0.724   
0.20 0.0       2397  287375  415103  15848165      0.008        0.006   
     1.0       6326  289311  411174  15846229      0.021        0.015   
     2.0      83630  313695  333870  15821845      0.210        0.200   
     3.0     187196  334388  230304  15801152      0.359        0.448   
     4.0     261527  348185  155973  15787355      0.429        0.626   
     5.0     326370  360026   91130  15775514      0.475        0.782   
0.25 0.0       3543  372376  413957  15763164      0.009        0.008   
     1.0       8902  375575  408598  15759965      0.023        0.021   
     2.0      93026  407281  324474  15728259      0.186        0.223   
     3.0     196412  431870  221088  15703670      0.313        0.470   
     4.0     270504  472920  146996  15662620      0.364        0.648   
     5.0     337436  488671   80064  15646869      0.408        0.808   

             Specificity  Accuracy    FDR     F1  
q    log2fc                                       
0.05 0.0           0.996     0.971  0.996  0.001  
     1.0           0.995     0.970  0.982  0.006  
     2.0           0.994     0.972  0.639  0.182  
     3.0           0.994     0.978  0.395  0.449  
     4.0           0.994     0.982  0.310  0.608  
     5.0           0.993     0.986  0.267  0.710  
0.10 0.0           0.994     0.969  0.993  0.003  
     1.0           0.994     0.969  0.974  0.010  
     2.0           0.993     0.972  0.632  0.214  
     3.0           0.993     0.977  0.421  0.469  
     4.0           0.992     0.982  0.346  0.613  
     5.0           0.992     0.985  0.305  0.709  
0.20 0.0           0.982     0.958  0.992  0.007  
     1.0           0.982     0.958  0.979  0.018  
     2.0           0.981     0.961  0.790  0.205  
     3.0           0.979     0.966  0.641  0.399  
     4.0           0.978     0.970  0.571  0.509  
     5.0           0.978     0.973  0.525  0.591  
0.25 0.0           0.977     0.952  0.991  0.009  
     1.0           0.977     0.953  0.977  0.022  
     2.0           0.975     0.956  0.814  0.203  
     3.0           0.973     0.961  0.687  0.376  
     4.0           0.971     0.963  0.636  0.466  
     5.0           0.970     0.966  0.592  0.543

per-sample (no aggregation) -- split method: insample_cv


TP     FP      FN        TN  Precision  Sensitivity  \
q    log2fc                                                            
0.05 0.0         16    661  312484  12098089      0.024        0.000   
     1.0        439    715  312061  12098035      0.380        0.001   
     2.0      55386   2906  257114  12095844      0.950        0.177   
     3.0     134765   5950  177735  12092800      0.958        0.431   
     4.0     187759   8081  124741  12090669      0.959        0.601   
     5.0     224932   9542   87568  12089208      0.959        0.720   
0.10 0.0         19   1022  312481  12097728      0.018        0.000   
     1.0        743   1156  311757  12097594      0.391        0.002   
     2.0      65384   6396  247116  12092354      0.911        0.209   
     3.0     144781  12952  167719  12085798      0.918        0.463   
     4.0     196423  17372  116077  12081378      0.919        0.629   
     5.0     234338  20640   78162  12078110      0.919        0.750   
0.20 0.0         37   1914  312463  12096836      0.019        0.000   
     1.0       1455   2549  311045  12096201      0.363        0.005   
     2.0      78153  17169  234347  12081581      0.820        0.250   
     3.0     156947  31581  155553  12067169      0.832        0.502   
     4.0     207204  40898  105296  12057852      0.835        0.663   
     5.0     247592  48593   64908  12050157      0.836        0.792   
0.25 0.0         60   2718  312440  12096032      0.022        0.000   
     1.0       1931   3878  310569  12094872      0.332        0.006   
     2.0      83333  24739  229167  12074011      0.771        0.267   
     3.0     161545  43793  150955  12054957      0.787        0.517   
     4.0     211420  56249  101080  12042501      0.790        0.677   
     5.0     253675  67401   58825  12031349      0.790        0.812   

             Specificity  Accuracy    FDR     F1  
q    log2fc                                       
0.05 0.0           1.000     0.975  0.976  0.000  
     1.0           1.000     0.975  0.620  0.003  
     2.0           1.000     0.979  0.050  0.299  
     3.0           1.000     0.985  0.042  0.595  
     4.0           0.999     0.989  0.041  0.739  
     5.0           0.999     0.992  0.041  0.822  
0.10 0.0           1.000     0.975  0.982  0.000  
     1.0           1.000     0.975  0.609  0.005  
     2.0           0.999     0.980  0.089  0.340  
     3.0           0.999     0.985  0.082  0.616  
     4.0           0.999     0.989  0.081  0.746  
     5.0           0.998     0.992  0.081  0.826  
0.20 0.0           1.000     0.975  0.981  0.000  
     1.0           1.000     0.975  0.637  0.009  
     2.0           0.999     0.980  0.180  0.383  
     3.0           0.997     0.985  0.168  0.626  
     4.0           0.997     0.988  0.165  0.739  
     5.0           0.996     0.991  0.164  0.814  
0.25 0.0           1.000     0.975  0.978  0.000  
     1.0           1.000     0.975  0.668  0.012  
     2.0           0.998     0.980  0.229  0.396  
     3.0           0.996     0.984  0.213  0.624  
     4.0           0.995     0.987  0.210  0.729  
     5.0           0.994     0.990  0.210  0.801

null (log2fc=0) per-sample FP calibration:


mean    max
split       q                    
insample_cv 0.05     66.10    173
            0.10    102.20    258
            0.20    191.40    508
            0.25    271.80    647
lobo        0.05   2755.68  11366
            0.10   3810.36  11738
            0.20  11495.00  34542
            0.25  14895.04  35587

In [11]:
import matplotlib.pyplot as plt

_PLOT_SPLITS = ["insample_cv", "lobo"]
_METRICS = ["Sensitivity", "Precision", "FDR", "Specificity"]
_LINESTYLES = {"Sensitivity": "-", "Precision": "--", "FDR": ":", "Specificity": "-."}
_MARKERS = {"Sensitivity": "o", "Precision": "s", "FDR": "^", "Specificity": "D"}

# grayscale gradient from black -- edit this list to restyle
PALETTE = {"Sensitivity": "#000000", "Precision": "#1916BB", "FDR": "#5250EB", "Specificity": "#9896F3"}

_plot_df = summary_ps.reset_index()
_plot_df = _plot_df[(_plot_df.split.isin(_PLOT_SPLITS)) & (_plot_df.log2fc > 0)]
_qs = sorted(_plot_df.q.unique())

fig, axes = plt.subplots(len(_qs), len(_PLOT_SPLITS), figsize=(5 * len(_PLOT_SPLITS), 2.5 * len(_qs)),
                         sharex=False, sharey=True, squeeze=False)
for i, q in enumerate(_qs):
    for j, sm in enumerate(_PLOT_SPLITS):
        ax = axes[i, j]
        sub = _plot_df[(_plot_df.q == q) & (_plot_df.split == sm)].sort_values("log2fc")
        for metric in _METRICS:
            ax.plot(sub.log2fc, sub[metric], color=PALETTE[metric], linestyle=_LINESTYLES[metric],
                    marker=_MARKERS[metric], label=metric)
        ax.axhline(q, color='red', linestyle=":", linewidth=1,
                  label="FDR target (q)" if (i, j) == (0, 0) else None)
        ax.set_ylim(-0.00, 1.05)
        if i == 0:
            if sm == "insample_cv":
                ax.set_title("Insample CV")
            elif sm == "lobo":
                ax.set_title("Unseen Batch (LOBO)")
        if j == 0:
            ax.set_ylabel(f"q={q}")
        if i == len(_qs) - 1:
            ax.set_xlabel("log2fc")
        ax.grid(alpha=0.2)
axes[2, -1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
plt.tight_layout()
plt.savefig("./DetectionLimitResults/per_sample_sweep.png", dpi=300)